# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Units Sold: {total_units:,}")

Total Revenue: $8,520.00
Total Units Sold: 783


The total revenue that is generated across all 400 orders is $8,520.00, representing a total volume of 783 units sold across all vendor transactions.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category', as_index=False)['revenue'].sum()

by_category = by_category.sort_values(by='revenue', ascending=False).reset_index(drop=True)

by_category['share_of_total'] = (by_category['revenue'] / df['revenue'].sum()) * 100

display_df = by_category.copy()
display_df['share_of_total'] = display_df['share_of_total'].map('{:.1f}%'.format)
display_df['revenue'] = display_df['revenue'].map('${:,.2f}'.format)

display(display_df)

,category,revenue,share_of_total
0,Food,"$4,293.00",50.4%
1,Merch,"$1,771.50",20.8%
2,Drink,"$1,554.00",18.2%
3,RainGear,$901.50,10.6%


Food generated the highest total revenue at \$4,293.00, which accounts for 50.4% of all sales. Merch followed at \$1,771.50 (20.8%), Drink at \$1,554.00 (18.2%), and RainGear generated the lowest revenue at \$901.50 (10.6%).

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendor_stats = df.groupby('vendor_id')['revenue'].agg(['mean', 'count']).reset_index()

vendor_stats.columns = ['vendor_id', 'avg_revenue', 'order_count']

vendor_stats = vendor_stats.sort_values(by='avg_revenue', ascending=False).reset_index(drop=True)

display_df = vendor_stats.copy()
display_df['avg_revenue'] = display_df['avg_revenue'].map('${:,.2f}'.format)
display(display_df)

,vendor_id,avg_revenue,order_count
0,V-01,$22.60,94
1,V-18,$21.75,108
2,V-05,$20.58,93
3,V-10,$20.31,105


Vendor V-01 generated the highest average order revenue at \$22.60 across 94 orders. V-18 averaged \$21.75 across 108 orders, V-05 averaged \$20.58 across 93 orders, and V-10 had the lowest average order revenue at \$20.31 across 105 orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_rev = df[df['category'] == 'Merch']['revenue'].sum()
total_rev = df['revenue'].sum()

merch_share = (merch_rev / total_rev) * 100

print(f"Merch Revenue: {merch_share:.1f}%")

Merch Revenue: 20.8%


Merch accounts for 20.8% of the total revenue which is about \$1,771.50 out of the total \$8,520.00 across all orders.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [9]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown / Unmatched Vendor')

print(f"Original Row Count: {len(df)} | Merged Row Count: {len(joined)}")
print(f"Original Revenue: ${df['revenue'].sum():,.2f} | Merged Revenue: ${joined['revenue'].sum():,.2f}")

unmatched_vendor = df[~df['vendor_id'].isin(vendor_names['vendor_id'])]['vendor_id'].unique()[0]
unmatched_count = (joined['vendor_id'] == unmatched_vendor).sum()

print(f"Unmatched Vendor ID: {unmatched_vendor} ({unmatched_count} orders)")

Original Row Count: 400 | Merged Row Count: 400
Original Revenue: $8,520.00 | Merged Revenue: $8,520.00
Unmatched Vendor ID: V-18 (108 orders)


**The unmatched vendor, and what I did about it:** _..._
Vendor V-18 is present in the orders dataset (108 orders) but missing from the vendor_names lookup table. Doing a left join made sure all 400 original rows and total revenue of \$8,520.00 were not dropped from the data. I assigned 'Unknown / Unmatched Vendor' to V-18 so its orders remain accounted for.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [10]:
pivot_df = pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_id',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

display_pivot = pivot_df.copy().fillna(0)
for col in display_pivot.columns:
    display_pivot[col] = display_pivot[col].map('${:,.2f}'.format)

display(display_pivot)

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,$171.00,"$1,338.00",$373.50,$241.50,"$2,124.00"
V-05,$298.50,$882.00,$489.00,$244.50,"$1,914.00"
V-10,$502.50,"$1,054.50",$400.50,$175.50,"$2,133.00"
V-18,$582.00,"$1,018.50",$508.50,$240.00,"$2,349.00"
Total,"$1,554.00","$4,293.00","$1,771.50",$901.50,"$8,520.00"


Across all vendors, Food generated the highest overall revenue at \$4,293.00, while RainGear generated the lowest at \$901.50. Vendor V-18 had the highest individual vendor revenue total at \$2,349.00 across categories, while V-05 had the lowest total revenue at \$1,914.00.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [11]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell these vendors that in order to optimize their sales, they should focus on inventory toward food items, which had the most total sales of $4,293.00 (50.4% of total revenue), showing a high overall demand. I also would tell them to have less stock on rain gear, since they only generated \$901.50 (10.6% of total revenue). This suggests that not many people buy rain gear at these vendors during games. Additionally, I think that individual vendors should adjust their strategies. In particular, vendors like V-05 had the lowest revenue at \$1,914.00 despite having 93 orders, having an average order value of only \$20.58. They should fix their pricing in order to increase their average order value toward the pricing of V-01 (\$22.60 per order).

b) I think the least trustworthy answer was to question 5. Vendor V-18 generated the highest number of orders at 108 and had the largest total revenue of $2,349.00 in the entire dataset. However, it was not there on the lookup table. While assigning a "Unmatched Vendor" with a left join prevented data loss, a percentage of orders still could not be matched to a specific business. I think that the report is less reliable because the top performing vendor is unknown, making it harder to make the most accurate financial decisions.